In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import math

In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import re

In [ ]:
import random

In [ ]:
import tldextract

In [ ]:
from urllib.parse import urlparse

In [ ]:
from scipy.stats import randint, uniform,entropy

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [ ]:
from xgboost import XGBClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score,accuracy_score

In [ ]:
from tqdm import tqdm

In [ ]:
import optuna

In [ ]:
from collections import Counter

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

In [ ]:
new_df = pd.DataFrame()

In [ ]:
ttld_df = pd.DataFrame()

In [ ]:
special_chars = set() 

In [ ]:
ip_pattern = r'[0-9]+(?:\.[0-9]+){3}'

In [ ]:
sus_words = ['secure', 'account', 'update', 'login', 'verify' ,'signin', 'bank',
            'notify', 'click', 'inconvenient']

In [ ]:
def find_special_char(x):
    special_chars_in_x = re.findall(r'[^a-zA-Z0-9]', x)
    special_chars.update(special_chars_in_x)
    return None

In [ ]:
def urlentropy(url):
    frequencies = Counter(url)
    prob = [frequencies[char] / len(url) for char in url]
    return entropy(prob, base=2)

In [ ]:
def redirection(url):
  pos = url.rfind('//') 
  return pos>7

In [ ]:
url_df = pd.read_csv('Dataset/dataset_urls.csv')

In [ ]:
url_df['URL'].str.strip()

In [ ]:
url_df

In [ ]:
url_df.head()

In [ ]:
url_df.sample(11)

In [ ]:
url_df_bad = url_df[url_df['Label'] == 'bad']

In [ ]:
url_df_bad['URL'].apply(find_special_char)

In [ ]:
special_chars = list(special_chars)

In [ ]:
special_chars.remove('.')

In [ ]:
special_chars.remove('/')

In [ ]:
new_df['Special Character'] = special_chars

In [ ]:
new_df['Frequency in bad URLs'] = new_df['Special Character'].apply(lambda x: url_df_bad[url_df_bad['URL'].str.contains(re.escape(x), regex=True)].shape[0])

In [ ]:
new_df['Bad probability'] = new_df['Frequency in bad URLs']/new_df['Special Character'].apply(lambda x: url_df[url_df['URL'].str.contains(re.escape(x), regex=True)].shape[0])

In [ ]:
new_df['Score'] = new_df['Bad probability']*new_df['Frequency in bad URLs'].apply(math.log)

In [ ]:
new_df.sort_values(by='Score', ignore_index=True, ascending=False, inplace=True)

In [ ]:
new_df

In [ ]:
dangerous_chars = list(new_df['Special Character'].head(10))

In [ ]:
print(dangerous_chars)

In [ ]:
plt.bar(new_df['Special Character'].head(10), new_df['Score'].head(10), color = 'cyan')
plt.xlabel('Special Character')
plt.ylabel('Score')
plt.show()

In [ ]:
ttld_list = pd.Series(url_df_bad['URL'].apply(lambda x: tldextract.extract(x).suffix)).unique()

In [ ]:
ttld_df['ttld'] = ttld_list

In [ ]:
ttld_df['Frequency in bad URLs'] = ttld_df['ttld'].apply(lambda x: url_df_bad[url_df_bad['URL'].str.contains(re.escape(x), regex=True)].shape[0])

In [ ]:
ttld_df['Bad probability'] = ttld_df['Frequency in bad URLs']/ttld_df['ttld'].apply(lambda x: url_df[url_df['URL'].str.contains(re.escape(x), regex=True)].shape[0])

In [ ]:
ttld_df['Score'] = ttld_df['Bad probability']*ttld_df['Frequency in bad URLs'].apply(math.log)

In [ ]:
ttld_df.sort_values(by='Score', ignore_index=True, ascending=False, inplace=True)

In [ ]:
ttld_df

In [ ]:
dangerous_ttlds = list(ttld_df['ttld'].head(10))

In [ ]:
print(dangerous_ttlds)

In [ ]:
plt.bar(ttld_df['ttld'].head(10), ttld_df['Score'].head(10), color = 'darkred')
plt.xlabel('Dangerous ttld')
plt.ylabel('Score')
plt.show()

In [ ]:
url_df['URL length'] = url_df['URL'].apply(len)

In [ ]:
url_df['Number of dots'] = url_df['URL'].apply(lambda x: x.count('.'))

In [ ]:
url_df['Number of slashes'] = url_df['URL'].apply(lambda x: x.count('/'))

In [ ]:
url_df['Percentage of numerical characters'] = url_df['URL'].apply(lambda x: sum(c.isdigit() for c in x))/url_df['URL length']


In [ ]:
url_df['Dangerous characters'] = url_df['URL'].apply(lambda x: any(char in x for char in dangerous_chars))

In [ ]:
url_df['Dangerous ttld'] = url_df['URL'].apply(lambda x: tldextract.extract(x).suffix in dangerous_ttlds)

In [ ]:
url_df['Entropy'] = url_df['URL'].apply(urlentropy)

In [ ]:
url_df['IP Address'] = url_df['URL'].apply(lambda x: bool(re.search(ip_pattern, x)))

In [ ]:
url_df['Domain name length'] = url_df['URL'].apply(lambda x: len(tldextract.extract(x).domain))

In [ ]:
url_df['Suspicious keywords'] = url_df['URL'].apply(lambda x: sum([word in x for word in sus_words]) != 0)

In [ ]:
url_df['Repetitions'] = url_df['URL'].apply(lambda x: True if re.search(r'(.)\1{2,}', tldextract.extract(x).domain) else False)

In [ ]:
url_df['Redirections'] = url_df['URL'].apply(redirection)

In [ ]:
url_df.head()

In [ ]:
scaler = MinMaxScaler()

In [ ]:
num_columns = ['URL length', 'Number of dots', 'Number of slashes', 'Domain name length', 'Entropy']

In [ ]:
url_df[num_columns] = scaler.fit_transform(url_df[num_columns])

In [ ]:
url_df['IP Address'] = url_df['IP Address'].astype(int)

In [ ]:
url_df['Suspicious keywords'] = url_df['Suspicious keywords'].astype(int)

In [ ]:
url_df['Repetitions'] = url_df['Repetitions'].astype(int)

In [ ]:
url_df['Redirections'] = url_df['Redirections'].astype(int)

In [ ]:
url_df['Dangerous characters'] = url_df['Dangerous characters'].astype(int)

In [ ]:
url_df['Dangerous ttld'] = url_df['Dangerous ttld'].astype(int)

In [ ]:
url_df['Label'] = (url_df['Label'] == 'good').astype(int)

In [ ]:
url_df.drop(columns=['URL'], inplace=True)

In [ ]:
url_df

In [ ]:
corr_matrix = url_df.corr()

In [ ]:
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', linewidths=0.5, annot_kws={"size": 6})
plt.show()
sns.heatmap(corr_matrix[['Label']].sort_values(by='Label').T, annot=True, cmap='coolwarm', linewidths=0.5, annot_kws={"size": 8})
plt.show()


In [ ]:
print(corr_matrix[['Label']].sort_values(by='Label'))

In [ ]:
pca = PCA(n_components=1)

In [ ]:
url_df['Entropy and length (PCA)'] = pca.fit_transform(url_df[['Entropy', 'URL length']])

In [ ]:
url_df.drop(columns=['Entropy', 'URL length'], inplace=True)

In [ ]:
url_df

In [ ]:
url_df['Label'].value_counts(normalize=True)

In [ ]:
n_samples = url_df['Label'].value_counts()[0]

In [ ]:
url_df_good = url_df[url_df['Label'] == 1]

In [ ]:
url_df_bad = url_df[url_df['Label'] == 0]

In [ ]:
url_df_goodsample = url_df_good.sample(n=n_samples, random_state=22)

In [ ]:
url_df_goodmissing = url_df_good.drop(url_df_goodsample.index)

In [ ]:
url_df = pd.concat([url_df_bad, url_df_goodsample], ignore_index=True)

In [ ]:
url_df

In [ ]:
y = url_df['Label']

In [ ]:
url_df.drop(columns=['Label'], inplace=True)

In [ ]:
url_df_train, url_df_test, y_train, y_test = train_test_split(url_df, y, test_size=0.2, random_state=22)

In [ ]:
y_goodmissing = url_df_goodmissing['Label']

In [ ]:
url_df_goodmissing.drop(columns=['Label'], inplace=True)

In [ ]:

url_df_test = pd.concat([url_df_test, url_df_goodmissing], axis=0)

In [ ]:
y_test = pd.concat([y_test, y_goodmissing], axis=0)

In [ ]:
kf = KFold(n_splits=3, shuffle=True, random_state=22)

In [ ]:
xgb_model = XGBClassifier(random_state=22)

In [ ]:
print(cross_val_score(xgb_model, url_df_train, y_train, cv=kf, scoring='accuracy').mean())

In [ ]:
rf_model = RandomForestClassifier(random_state=22)

In [ ]:
print(cross_val_score(rf_model, url_df_train, y_train, cv=kf, scoring='accuracy').mean())

In [ ]:
rf_model.fit(url_df_train, y_train)

In [ ]:
importances = rf_model.feature_importances_

In [ ]:
feature_names = url_df.columns 

In [ ]:
indices = np.argsort(importances)[::-1]

In [ ]:
plt.title('Feature Importance (RandomForestClassifier)')
plt.bar(range(url_df.shape[1]), importances[indices], align='center', color="blue")
plt.xticks(range(url_df.shape[1]), feature_names[indices], rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.show()

In [ ]:
xgb_model.fit(url_df_train, y_train)

In [ ]:
importances = xgb_model.feature_importances_

In [ ]:
feature_names = url_df.columns 

In [ ]:
indices = np.argsort(importances)[::-1]

In [ ]:
plt.title('Feature Importance (url_dfGBClassifier)')
plt.bar(range(url_df.shape[1]), importances[indices], align='center')
plt.xticks(range(url_df.shape[1]), feature_names[indices], rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.show()

In [ ]:
rf_pred = rf_model.predict(url_df_test)

In [ ]:
xgb_pred = xgb_model.predict(url_df_test)

In [ ]:
print(accuracy_score(y_test, rf_pred))

In [ ]:
print(accuracy_score(y_test, xgb_pred))

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 400)
    max_depth = trial.suggest_int('max_depth', 3, 7)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-3, 0.3)
    subsample = trial.suggest_uniform('subsample', 0.6, 1.0)
    reg_alpha = trial.suggest_loguniform('reg_alpha', 1e-3, 10.0)
    model = XGBClassifier(
        random_state=22,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        reg_alpha=reg_alpha,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )

    
    mean_score = cross_val_score(model, url_df, y, cv=kf, scoring='accuracy').mean()
    return mean_score

In [ ]:
study = optuna.create_study(direction='maximize')

In [ ]:
study.optimize(objective, n_trials=50) 

In [ ]:
print("Best hyperparameters:", study.best_params)
print("Best accuracy:", study.best_value)

In [ ]:
best_xgb_model = XGBClassifier(random_state=22, **study.best_params)

In [ ]:
best_xgb_model.fit(url_df_train, y_train)

In [ ]:
best_xgb_pred = best_xgb_model.predict(url_df_test)

In [ ]:
print(accuracy_score(y_test, best_xgb_pred))